# Musanga V3 / 02 — Fast first U-Net pilot

**Purpose:** learn whether the UAV RGB + new labels contain usable predictive signal before launching full CV. Two small controlled models: (A) exhaustive circles only; (B) exhaustive + partial tree crowns. **Same within-site spatial CV fold 0**, same validation tiles, and no training on test circles. No full-mosaic inference. Pilot metrics are *diagnostic*, not final unbiased performance estimates.

Run Notebook 01 through Block 10 first. Runtime: GPU recommended. Prints live epoch progress. Native RGB data unchanged.


## 01 — GPU, inputs and spatial CV manifest


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02_pilot';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261015;CROP=256;EPOCHS=3;BATCH=8;MAX_PER_SITE_KIND=35
STEPS_PER_EPOCH=80
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)



## 02 — Standalone window reference generator

Reuses Notebook 01's label policy. Geometries repaired in memory. Exhaustive holdout circles never contribute to training, and pixels outside the general ROI are ignored. The pilot uses **native-resolution 256-pixel center crops** of Notebook 01's window candidates, and reads only those small raster regions.


In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))
def make_masks(name,window,role,include_partial):
    d=geo[name];bounds=box(*rasterio.windows.bounds(window,d['transform']))
    affine=rasterio.windows.transform(window,d['transform'])
    size=(int(window.height),int(window.width))
    def burn(series):
        gs=[g.intersection(bounds) for g in series if g is not None and g.intersects(bounds)]
        return rasterize([(g,1) for g in gs if not g.is_empty],out_shape=size,
            transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros(size,bool)
    model_roi=burn(d['roi'].geometry)
    cs=d['circles']
    selected=cs[(cs.fold!=FOLD) if role=='train' else (cs.fold==FOLD)]
    # Only the selected mosaic owns the exhaustive circle.
    selected=selected[selected.owner==name]
    full=burn(selected.geometry)&model_roi
    pos=burn(d['positive'].geometry)&full
    partpos=burn(d['trees'].loc[d['trees'].class01==1].geometry)
    partneg=burn(d['trees'].loc[d['trees'].class01==0].geometry)
    contradictions=(pos&partneg)|(full&~pos&partpos)|(~full&partpos&partneg)
    y=(pos|(~full&partpos)).astype('float32')
    if role=='val':
        weight=full
    else:
        weight=full | ((partpos|partneg)&~full if include_partial else np.zeros(size,bool))
        weight &= ~contradictions
        # All heldout circles in this forest group, even if assigned to another MOS mosaic
        heldout=cs[cs.fold==FOLD]
        blocked=burn([g.buffer(15.) for g in heldout.geometry])
        weight &= ~blocked
    weight &= model_roi
    return y,weight.astype('float32'),full,contradictions
print('Window mask generator ready. Original vector sources remain untouched.')


## 03 — Balanced small pilot subset and input QA

Sample from **four forest groups**, with at most 35 candidate windows per site/supervision type. Evaluate the same independent exhaustive holdout windows for both models. Screen out windows without enough usable labelled pixels, and check RGB validity. To keep the pilot fast, cap the in-memory array cache.


In [ ]:
def choose_windows(split,kind=None,cap_per_site=35):
    a=splits[(splits.cv_fold==FOLD)&(splits.split==split)].copy()
    if kind is not None:a=a[a.kind==kind]
    result=[]
    for site,g in a.groupby('site'):
        # Include diverse mosaics; reproducible within-site shuffle.
        g=g.sample(frac=1,random_state=SEED)
        result.extend(g.iloc[:min(cap_per_site,len(g))].to_dict('records'))
    return result
training=choose_windows('train',None,MAX_PER_SITE_KIND*2)
validation=choose_windows('val','exhaustive',18)
print('Candidate windows: training',len(training),'validation',len(validation))
# Read native 256x256 center cutouts once into RAM.
def load_crop(rec,role,combined=True):
    name=rec['mosaic']
    row=int(rec['row'])+max(0,(int(rec['height'])-CROP)//2)
    col=int(rec['col'])+max(0,(int(rec['width'])-CROP)//2)
    if int(rec['height'])<CROP or int(rec['width'])<CROP:return None
    window=Window(col,row,CROP,CROP)
    y,w,full,unc=make_masks(name,window,role,include_partial=combined)
    if role=='train' and float(w.sum())<500:return None
    if role=='val' and float(w.sum())<200:return None
    with rasterio.open(geo[name]['path']) as ds:
        rgb=ds.read([1,2,3],window=window,masked=True)
    band_ok=~np.any(np.ma.getmaskarray(rgb),axis=0)
    raw=rgb.filled(0).astype('float32')
    rgb_ok=band_ok&np.any(raw!=0,axis=0)&np.any(raw!=255,axis=0)
    # Inference must use the original data validity; unknown pixels never train.
    w *= rgb_ok
    if float(w.sum())<200:return None
    rgb=np.clip(raw/255.,0,1).transpose(1,2,0)
    return dict(rgb=rgb.astype('float32'),y=y,w=w.astype('float32'),
                full=full,unc=unc,site=geo[name]['site'],name=name,
                circle_id=int(rec['circle_id']),kind=rec['kind'])
train_cache=[v for x in training if (v:=load_crop(x,'train',combined=True)) is not None]
val_cache=[v for x in validation if (v:=load_crop(x,'val',combined=True)) is not None]
assert train_cache and val_cache,'Insufficient valid windows: inspect ROI/masks'
print('Usable RGB windows:',len(train_cache),'train |',len(val_cache),'holdout')
print('Train per site:',pd.Series([x['site'] for x in train_cache]).value_counts().to_dict())
print('Holdout per site:',pd.Series([x['site'] for x in val_cache]).value_counts().to_dict())
print('Training labelled fraction median:',round(float(np.median([x['w'].mean() for x in train_cache])),3))
assert set(x['site'] for x in val_cache)==set(['Baego','MOS','Yangambi','Yoko']), 'Need validation coverage from every site'



## 04 — Compact U-Net and masked loss

Pilot trains on RGB only. Standard flip/rotation augmentation; **no illumination augmentation yet** so we first measure baseline signal. Training masks exclude all explicit label disagreements. Validation uses full exhaustive ground truth.


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')


## 05 — First two small models: exhaustive-only vs combined

Both models start from identical weights and use the same train/validation windows. Logs stream after every epoch. Only **three epochs** by default. These are pilot diagnostics, not finalized CV models.


In [ ]:
def run_pilot(label,combined):
    torch.manual_seed(SEED)
    model=TinyUNet().to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
    # To keep A/B comparable, use identical RGB windows; modify only loss weights.
    rows=[]
    for d in train_cache:
        q=dict(d)
        if not combined:
            # Recompute training masks for the same RGB crop for exhaustive-only variant.
            # Exhaustive source support = circle pixels; uncertain pixels ignored.
            q['w']=(d['full'] & ~d['unc']).astype('float32')*d['w']
        rows.append(q)
    rows=[d for d in rows if d['w'].sum()>100]
    print(label,'usable training chips:',len(rows),flush=True)
    assert len(rows)>4, f'{label}: insufficient supervised windows'
    loader=DataLoader(CropData(rows,augment=True),batch_size=BATCH,shuffle=True,num_workers=0)
    for epoch in range(EPOCHS):
        model.train();losses=[];t0=time.time()
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            optimizer.zero_grad(set_to_none=True)
            pred=model(x)
            loss=masked_loss(pred,y,w)
            loss.backward();optimizer.step()
            losses.append(float(loss.item()))
        print(f'{label} epoch {epoch+1}/{EPOCHS} | masked BCE {np.mean(losses):.4f} | {time.time()-t0:.1f}s',flush=True)
    model.eval()
    preds=[]
    with torch.no_grad():
        for d in val_cache:
            x=torch.from_numpy(d['rgb'].transpose(2,0,1).copy())[None].to(device)
            p=torch.sigmoid(model(x))[0].cpu().numpy()
            preds.append(p)
    torch.save(model.state_dict(),OUT/f'02_pilot_{label}.pt')
    return model,preds
t0=time.time()
model_A,pred_A=run_pilot('exhaustive_only',False)
model_B,pred_B=run_pilot('combined',True)
print('Total pilot training seconds:',round(time.time()-t0,1))



## 06 — Preliminary held-out diagnostics

Metrics use **valid RGB + exhaustive pixels only**. Circle coverage in this quick preview is sampled, not complete. Evaluate probability mean versus observed cover; report pixel AUC/AP/Brier and thresholded F1 for context. Empty single-class windows are excluded from AUC/AP computation, not from cover bias.


In [ ]:
def evaluate(label,predictions):
    rows=[]
    for d,p in zip(val_cache,predictions):
        mask=d['w']>0
        truth=d['y'][mask].astype('uint8');prob=p[mask]
        if len(truth)<200:continue
        cover_true=100*float(truth.mean());cover_prob=100*float(prob.mean())
        binary=prob>=.5
        tp=int(((binary==1)&(truth==1)).sum())
        fp=int(((binary==1)&(truth==0)).sum())
        fn=int(((binary==0)&(truth==1)).sum())
        rows.append(dict(model=label,site=d['site'],circle_id=d['circle_id'],n_pixels=len(truth),
            observed_cover_pct=cover_true,mean_probability_cover_pct=cover_prob,
            cover_bias_pp=cover_prob-cover_true,
            brier=float(np.mean((prob-truth)**2)),
            f1_05=2*tp/max(2*tp+fp+fn,1),
            ap=average_precision_score(truth,prob) if len(np.unique(truth))>1 else np.nan,
            auc=roc_auc_score(truth,prob) if len(np.unique(truth))>1 else np.nan))
    return pd.DataFrame(rows)
scores=pd.concat([evaluate('exhaustive_only',pred_A),evaluate('combined',pred_B)],ignore_index=True)
display(scores.groupby(['model','site']).agg(
    sampled_windows=('circle_id','size'),
    mean_cover_bias_pp=('cover_bias_pp','mean'),
    cover_mae_pp=('cover_bias_pp',lambda s:float(np.mean(np.abs(s)))),
    brier=('brier','mean'),f1_05=('f1_05','mean'),ap=('ap','mean'),auc=('auc','mean')).round(3))
scores.to_csv(OUT/'02_pilot_heldout_window_scores.csv',index=False)
display(scores.groupby('model').agg(n_windows=('circle_id','size'),
    cover_mae_pp=('cover_bias_pp',lambda s:float(np.abs(s).mean())),
    brier=('brier','mean'),ap=('ap','mean'),auc=('auc','mean')).round(3))
print('DIAGNOSTIC ONLY: small subset / short training; not full-circle, full-fold CV.')



## 07 — Visual comparison on independent held-out windows


In [ ]:
fig,axs=plt.subplots(min(4,len(val_cache)),4,figsize=(13,3.1*min(4,len(val_cache))),constrained_layout=True)
if axs.ndim==1:axs=axs[None,:]
chosen=[]
for site in ['Baego','MOS','Yangambi','Yoko']:
    matches=[i for i,d in enumerate(val_cache) if d['site']==site]
    if matches:chosen.append(matches[len(matches)//2])
for row_idx,j in enumerate(chosen[:len(axs)]):
    d=val_cache[j]
    axs[row_idx,0].imshow(d['rgb']);axs[row_idx,0].set_title(d['site']+' RGB')
    truth=np.where(d['w']>0,d['y'],np.nan)
    axs[row_idx,1].imshow(truth,vmin=0,vmax=1,cmap='Greens');axs[row_idx,1].set_title('Exhaustive reference')
    axs[row_idx,2].imshow(np.where(d['w']>0,pred_A[j],np.nan),vmin=0,vmax=1,cmap='viridis')
    axs[row_idx,2].set_title('A: exhaustive only')
    im=axs[row_idx,3].imshow(np.where(d['w']>0,pred_B[j],np.nan),vmin=0,vmax=1,cmap='viridis')
    axs[row_idx,3].set_title('B: combined')
for ax in axs.flat:ax.axis('off')
fig.colorbar(im,ax=axs[:,2:].ravel().tolist(),shrink=.65,label='Musanga probability')
fig.savefig(OUT/'02_pilot_spatial_comparison.png',dpi=150,bbox_inches='tight')
display(fig);plt.close(fig)
print('Pilot outputs:',OUT)

